# Gen Relevant Topics

* Given a bunch of knowledge base, and user specified topics --> for each topic, find the most relevant text in each documents
  * rank the retrieved text of needed

In [1]:
import os
from dotenv import load_dotenv
from chonkie import SemanticChunker

load_dotenv("../.env")  # loads OPENAI_API_KEY from .env file


True

In [2]:
selected_docs = ['0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt',
                '0a7f058fad69b8be410fed4859f3be4b4fc5d55cf631e4f0f3409985f78d26bb.txt',
                '0a0bcc5de9b3f7f28b94f8a23b2d620eb4372a61fbcb69604993123bc59fc2a3.txt',
                'fffe1b498f9816a054db6841ad9bf85cdc18bb36cba5b62f6275f1b386d50be3.txt',
]
wix_docs_path = '../datasets/wix_docs'

for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    token_count = len(text) // 4
    print(f"{doc}: ~{token_count} tokens (estimated at 4 chars/token)")


0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt: ~1262 tokens (estimated at 4 chars/token)
0a7f058fad69b8be410fed4859f3be4b4fc5d55cf631e4f0f3409985f78d26bb.txt: ~1067 tokens (estimated at 4 chars/token)
0a0bcc5de9b3f7f28b94f8a23b2d620eb4372a61fbcb69604993123bc59fc2a3.txt: ~80 tokens (estimated at 4 chars/token)
fffe1b498f9816a054db6841ad9bf85cdc18bb36cba5b62f6275f1b386d50be3.txt: ~348 tokens (estimated at 4 chars/token)


## Chunking

* Langchain basic text spliters: https://docs.langchain.com/oss/python/integrations/splitters
* Chonkie chunkers: https://docs.chonkie.ai/oss/chunkers/overview

#### Semantic Chunking

* `pip install "chonkie[semantic]"` available up to python 3.13, doesn't support newer python
* `pip install sentence-transformers`
* `pip install "chonkie[neural]"`

* Oberservations 🍀
  * Chonkie's AutoEmbeddings, looks like allows different embedding models but for openAI, somehow it will load from HuggingFace and gets error, so finally I had to use OpenAIEmbeddings directly from chonkie.
  * `SemanticChunker` uses the embedding model's tokenizer internally for token counting, and chonkie's OpenAIEmbeddings tokenizer wrapper isn't compatible with it. Passing a sentence-transformer model name string (like "all-MiniLM-L6-v2") works correctly and is what chonkie recommends for semantic chunking. 

In [10]:
chunker = SemanticChunker(
    embedding_model='all-MiniLM-L6-v2',          # sentence-transformer model, decide where to split
    threshold=0.8,                               # Similarity threshold (0-1)
    chunk_size=1200,                             # Maximum tokens per chunk
    similarity_window=3,                         # Window for similarity calculation
    skip_window=0                                # Skip-and-merge window (0=disabled)
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9207.05it/s]
/Users/hanhanwu/Documents/Github/Yokan/.venv13/lib/python3.13/site-packages/chonkie/embeddings/sentence_transformer.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()


In [11]:
def _print_wrapped(text, words_per_line=50):
    if not text:
        return
    words = str(text).split()
    for i in range(0, len(words), words_per_line):
        print(' '.join(words[i:i+words_per_line]))

for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = chunker.chunk(text)
    print(f"{doc}: {len(chunks)} chunks")
    for chunk in chunks:
        _print_wrapped(chunk, 25)
        print("----CHUNK SEPARATOR----")
    print("--------------------------DOC SEPARATOR------------------------")

0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt: 16 chunks
Facebook Ads: Creating a New Campaign Create targeted campaigns for Facebook and Instagram and get your business in front of thousands of people. Your Facebook
Ads campaign is entirely in your control: design your ad, choose a monthly budget and set a specific target audience.
----CHUNK SEPARATOR----
Wix's AI also creates lookalike audiences to help get more eyes on your business.Before you begin:Make sure you've already done the following: Connected your Facebook
Page Upgraded to a Premium Plan Published your site Connected a domainLearn how get your site ready to launch a Facebook Ads campaign.Step 1 |
Choose a campaign goalOnce you've connected your Facebook page to Wix, the first step of creating a campaign is choosing a campaign goal. Depending on
your business you can focus on getting more store orders, bookings to your site, or to increase views and traffic to a page.Important:Once you choose
a campa

#### Sentence Chunking

* `pip install chonkie` (included in base install)
* Splits text into chunks based on sentence boundaries, respecting token limits

In [ ]:
from chonkie import SentenceChunker

# Basic initialization with default parameters
sentence_chunker = SentenceChunker(
    tokenizer="character",     # Default tokenizer (or use "gpt2", etc.)
    chunk_size=2048,           # Maximum tokens per chunk
    chunk_overlap=128,         # Overlap between chunks
    min_sentences_per_chunk=1  # Minimum sentences in each chunk
)

In [ ]:
for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = sentence_chunker.chunk(text)
    print(f"{doc}: {len(chunks)} chunks")
    for chunk in chunks:
        _print_wrapped(chunk, 25)
        print("----CHUNK SEPARATOR----")
    print("--------------------------DOC SEPARATOR------------------------")

#### Neural Chunking

* `pip install "chonkie[neural]"`
* Uses a fine-tuned ML model to detect semantically meaningful chunk boundaries

In [ ]:
from chonkie import NeuralChunker

# Basic initialization with default parameters
neural_chunker = NeuralChunker(
    model="mirth/chonky_modernbert_base_1",  # Default model
    device_map="cpu",                        # Device to run the model on ('cpu', 'cuda', etc.)
    min_characters_per_chunk=10,             # Minimum characters for a chunk
)

In [ ]:
for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = neural_chunker.chunk(text)
    print(f"{doc}: {len(chunks)} chunks")
    for chunk in chunks:
        _print_wrapped(chunk, 25)
        print("----CHUNK SEPARATOR----")
    print("--------------------------DOC SEPARATOR------------------------")